# Trastuzumab B-chain hinge remodel (RunPod Jupyter)

Port of `remodeling.ipynb`. Goal: FastRelax B220–235 so the B-chain hinge can open, then take the best PDB back into `adcsim`.

Flexible: B220–235 (bb + chi)  
Fixed anchors: B217–219 and B236–238

**Kernel menu has no `pyrosetta_env`:** that is normal on a new pod. Stay in the current kernel and run the next two cells. They will find an existing PyRosetta Python and register it, or install PyRosetta here.

In [ ]:
import os, sys, glob, subprocess
from pathlib import Path

print("This kernel Python:", sys.executable)
print("Version:", sys.version.split()[0])

candidates = [
    "/workspace/pyrosetta_env/bin/python",
    "/workspace/pyrosetta_env/bin/python3",
    str(Path.home() / "pyrosetta_env/bin/python"),
    "/opt/conda/envs/pyrosetta_env/bin/python",
    "/opt/conda/bin/python",
]
candidates += glob.glob("/workspace/*/bin/python")
candidates += glob.glob("/opt/conda/envs/*/bin/python")
candidates += glob.glob("/root/miniconda*/envs/*/bin/python")

found = []
seen = set()
for p in candidates:
    if not p or p in seen or not os.path.isfile(p):
        continue
    seen.add(p)
    try:
        r = subprocess.run(
            [p, "-c", "import pyrosetta, sys; print(sys.executable)"],
            capture_output=True, text=True, timeout=30,
        )
        if r.returncode == 0:
            found.append((r.stdout.strip() or p))
            print("FOUND pyrosetta:", p)
        else:
            print("no pyrosetta:", p)
    except Exception as e:
        print("skip", p, type(e).__name__)

try:
    import pyrosetta
    print("THIS kernel already has pyrosetta:", pyrosetta.__file__)
    found.insert(0, sys.executable)
except Exception:
    print("THIS kernel does NOT have pyrosetta")

FOUND_PYROSETTA = list(dict.fromkeys(found))
print("FOUND_PYROSETTA =", FOUND_PYROSETTA)

## Register kernel or install into this kernel

- If the previous cell printed `FOUND pyrosetta: ...` → this cell registers Jupyter kernel `pyrosetta_env`. Then **Kernel → Change kernel → pyrosetta_env**. If the list is still empty, close and reopen the notebook.
- If nothing was found → this cell installs the quarterly wheel into **this** kernel (academic/non-commercial, ~1–2 GB). Then **Kernel → Restart Kernel** and continue. Do not `pip install pyrosetta` from public PyPI.

In [ ]:
import subprocess, sys

FORCE_INSTALL = False  # True = reinstall into THIS kernel even if another Python has pyrosetta

def sh(cmd):
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)

if FOUND_PYROSETTA and not FORCE_INSTALL:
    py = FOUND_PYROSETTA[0]
    print("Using existing PyRosetta Python:", py)
    sh([py, "-m", "pip", "install", "-q", "ipykernel"])
    sh([py, "-m", "ipykernel", "install", "--user",
        "--name", "pyrosetta_env", "--display-name", "pyrosetta_env"])
    print()
    print("Kernel registered as: pyrosetta_env")
    print("Now: Kernel → Change kernel → pyrosetta_env")
    print("If still missing: close this notebook and open it again.")
    print("Then run the init cell (import pyrosetta).")
else:
    print("No existing PyRosetta found. Installing into THIS kernel:")
    print(sys.executable)
    print("Academic / non-commercial license. Wheel is ~1–2 GB.")
    sh([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
    sh([
        sys.executable, "-m", "pip", "install", "pyrosetta",
        "--find-links", "https://west.rosettacommons.org/pyrosetta/quarterly/release",
    ])
    sh([sys.executable, "-m", "pip", "install", "-q", "ipykernel"])
    sh([sys.executable, "-m", "ipykernel", "install", "--user",
        "--name", "pyrosetta_env", "--display-name", "pyrosetta_env"])
    print()
    print("Install done. Kernel → Restart Kernel, then run the init cell.")

In [ ]:
import sys
import pyrosetta
print("Python:", sys.executable)
print("pyrosetta:", pyrosetta.__file__)
pyrosetta.init("-mute all")
print("PyRosetta initialization: OK")

## Paths

Edit `WORK_DIR` if your files are not under `/workspace/trastuzumab_project`.

In [ ]:
from pathlib import Path

WORK_DIR = Path("/workspace/trastuzumab_project")
PDB_PATH = WORK_DIR / "trastuzumab_boltz_model_0.pdb"
OUT_DIR = WORK_DIR / "B220_235_rosetta"
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_CANDIDATES = 10
FLEX_START, FLEX_END = 220, 235
ANCHOR_N = range(217, 220)
ANCHOR_C = range(236, 239)

print("WORK_DIR :", WORK_DIR)
print("PDB      :", PDB_PATH, "exists=", PDB_PATH.exists())
print("OUT_DIR  :", OUT_DIR)
if not PDB_PATH.exists():
    raise FileNotFoundError(
        f"Put trastuzumab_boltz_model_0.pdb at {PDB_PATH} or change WORK_DIR"
    )

## FastRelax: 10 candidates on B220–235

Best-scoring pose is written to `B220_235_rosetta_best.pdb` — copy that file back into `adcsim`.

In [ ]:
from pyrosetta import pose_from_pdb
from pyrosetta.rosetta.core.pose import Pose
from pyrosetta.rosetta.core.kinematics import MoveMap
from pyrosetta.rosetta.core.scoring import ScoreFunctionFactory
from pyrosetta.rosetta.protocols.relax import FastRelax

pose = pose_from_pdb(str(PDB_PATH))
print("Pose residues:", pose.total_residue())

def pdb2pose_B(pdb_res):
    r = pose.pdb_info().pdb2pose("B", int(pdb_res))
    if r == 0:
        raise ValueError(f"B:{pdb_res} not found in pose")
    return r

print("B220 pose index:", pdb2pose_B(220), " B235:", pdb2pose_B(235))

movemap = MoveMap()
movemap.set_bb(False)
movemap.set_chi(False)

for pdb_res in range(FLEX_START, FLEX_END + 1):
    r = pdb2pose_B(pdb_res)
    movemap.set_bb(r, True)
    movemap.set_chi(r, True)

for pdb_res in list(ANCHOR_N) + list(ANCHOR_C):
    r = pdb2pose_B(pdb_res)
    movemap.set_bb(r, False)
    movemap.set_chi(r, False)

scorefxn = ScoreFunctionFactory.create_score_function("ref2015")
scorefxn.set_weight(pyrosetta.rosetta.core.scoring.fa_rep, 1.0)
scorefxn.set_weight(pyrosetta.rosetta.core.scoring.cart_bonded, 0.5)

print("=" * 70)
print("Backbone + side chains: B220–235")
print("Fixed anchors: B217–219 / B236–238")
print("=" * 70)

best_pose = None
best_score = None

for run_id in range(N_CANDIDATES):
    test_pose = Pose()
    test_pose.assign(pose)

    relax = FastRelax()
    relax.set_scorefxn(scorefxn)
    relax.set_movemap(movemap)
    relax.apply(test_pose)

    score = float(scorefxn(test_pose))
    cand_path = OUT_DIR / f"B220_235_fastrelax_{run_id + 1:02d}.pdb"
    test_pose.dump_pdb(str(cand_path))

    if best_score is None or score < best_score:
        best_score = score
        best_pose = Pose()
        best_pose.assign(test_pose)

    print(f"Candidate {run_id + 1:2d} | Score={score:10.3f} | {cand_path.name}")

out_path = OUT_DIR / "B220_235_rosetta_best.pdb"
best_pose.dump_pdb(str(out_path))

print("=" * 70)
print(f"Best score = {best_score:.3f}")
print(f"Saved: {out_path}")
print("Copy this PDB back. That is the only file adcsim needs from this notebook.")

## Optional: extract B217–238 as a local pose

Only needed for the torsion-sampling cell. FastRelax best PDB above is enough for `adcsim`.

In [ ]:
local_path = WORK_DIR / "B217_238_local.pdb"
full = pose_from_pdb(str(PDB_PATH))
start = full.pdb_info().pdb2pose("B", 217)
end = full.pdb_info().pdb2pose("B", 238)

local_pose = full.clone()
local_pose.delete_residue_range_slow(end + 1, local_pose.total_residue())
local_pose.delete_residue_range_slow(1, start - 1)
local_pose.dump_pdb(str(local_path))

print("Original residues:", full.total_residue())
print("Local residues   :", local_pose.total_residue(), "(expect 22)")
print("Saved:", local_path)

## Optional: local torsion sampling on B220–235

Skip unless FastRelax is not enough. Output is a 22-residue fragment — **do not** feed it to `adcsim`.

In [ ]:
RUN_TORSION = False

if not RUN_TORSION:
    print("skip torsion sampling")
else:
    import math
    from pyrosetta.rosetta.protocols.minimization_packing import MinMover

    tdir = WORK_DIR / "B220_235_rosetta_torsion"
    tdir.mkdir(parents=True, exist_ok=True)
    frag = pose_from_pdb(str(WORK_DIR / "B217_238_local.pdb"))
    print("Local pose residues:", frag.total_residue())

    sfxn = ScoreFunctionFactory.create_score_function("ref2015")
    mm = MoveMap()
    mm.set_bb(False)
    mm.set_chi(False)
    for i in range(4, 20):
        mm.set_bb(i, True)
        mm.set_chi(i, True)

    best_t = None
    best_t_score = float("inf")
    for cid in range(1, 11):
        cand = frag.clone()
        for i in range(4, 20):
            cand.set_phi(i, cand.phi(i) + 8.0 * math.sin(cid + i))
            cand.set_psi(i, cand.psi(i) + 8.0 * math.cos(cid + i))
        MinMover(mm, sfxn, "lbfgs_armijo_nonmonotone", 0.01, True).apply(cand)
        sc = float(sfxn(cand))
        op = tdir / f"B220_235_torsion_{cid:02d}.pdb"
        cand.dump_pdb(str(op))
        print(f"Candidate {cid:2d} | Score={sc:10.3f} | {op.name}")
        if sc < best_t_score:
            best_t_score = sc
            best_t = cand.clone()
    best_t.dump_pdb(str(tdir / "B220_235_torsion_best.pdb"))
    print("Best torsion score:", best_t_score)
    print("NOTE: 22-residue fragment. Do not use as adcsim antibody_pdb.")